# Exploratory Data Analysis (EDA) - Crop Risk DSS
This notebook provides an interactive exploratory data analysis of the weather features and the resulting risk probabilities across different cities.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Setup paths
data_dir = Path('../data/processed')


In [ ]:
# Load the engineered features and risk probabilities
features_df = pd.read_csv(data_dir / 'engineered_features.csv')
features_df['date'] = pd.to_datetime(features_df['date'])
features_df['DOY'] = features_df['date'].dt.dayofyear

risk_df = pd.read_csv(data_dir / 'risk_probabilities.csv')

print(f"Features Data Shape: {features_df.shape}")
print(f"Risk Data Shape: {risk_df.shape}")


## 1. Weather Feature Distributions
Let's analyze the distribution of Maximum Temperatures across different cities.

In [ ]:
fig = px.box(features_df, x='city_name', y='tmax', color='city_name',
             title='Maximum Temperature Distribution by City',
             labels={'tmax': 'Max Temperature (°C)', 'city_name': 'City'})
fig.show()


## 2. Annual Rainfall Patterns
Visualizing how rainfall is distributed over the year (Day of Year) for a specific city.

In [ ]:
faisalabad_df = features_df[features_df['city_name'] == 'Faisalabad'].groupby('DOY')['prcp'].mean().reset_index()

fig = px.line(faisalabad_df, x='DOY', y='prcp', 
              title='Average Daily Rainfall over the Year in Faisalabad',
              labels={'prcp': 'Average Precipitation (mm)', 'DOY': 'Day of Year'})
fig.update_traces(line_shape='spline', fill='tozeroy')
fig.show()


## 3. Crop Risk Probabilities
Let's explore the computed risks (Heat Stress, Frost, Heavy Rain) over the year.

In [ ]:
# Filter for 30-day window in a specific city
city_risk = risk_df[(risk_df['city_name'] == 'Okara') & (risk_df['window_days'] == 30)]

fig = px.line(city_risk, x='DOY', y='probability', color='risk_type',
              title='30-Day Risk Probabilities in Okara over the Year',
              labels={'probability': 'Probability', 'DOY': 'Day of Year', 'risk_type': 'Risk Type'})
fig.update_traces(line_shape='spline')
fig.show()


## 4. Correlation Analysis
Understanding the relationships between different weather features.

In [ ]:
numeric_cols = ['tmax', 'tmin', 'prcp', 'heat_stress_flag', 'frost_flag', 'heavy_rain_flag']
corr = features_df[numeric_cols].corr()

fig = px.imshow(corr, text_auto=True, aspect='auto', title='Feature Correlation Heatmap', color_continuous_scale='RdBu_r')
fig.show()
